# Assignment 1: Information Retrieval

## Overview
In this assignment you will implement a BM25 retrieval system from scratch.
Given a text query, your system must find the most relevant documents from the **Cranfield Collection** — 
a foundational corpus for evaluating information retrieval systems containing:
- **1,400** documents
- **225** queries
- **1,837** relevance judgments (qrels)

## Rules
- You may only use **built-in Python libraries** plus `numpy` and `pandas`
- Full-packaged retrieval solutions (e.g. `rank_bm25`) are **not permitted**
- Do **not** modify function signatures or the skeleton class
- Do **not** change cells marked **do not edit**

## Deadline
**March 19, 23:59 CET**

## How to Use This Notebook
1. Run imports and BM25 Skeleton class first — always
2. Implement each task in the cells marked **TODO: your code here** and answer questions marked **TODO**
3. If you restart the kernel, re-run all cells from top to bottom

## Load the data

In [1]:
import ir_datasets
dataset = ir_datasets.load("cranfield")

### Inspect the queries

In [2]:
for query in dataset.queries_iter():
    print(query) # namedtuple<query_id, text>
    break

GenericQuery(query_id='1', text='what similarity laws must be obeyed when constructing aeroelastic models\nof heated high speed aircraft .')


### Inspect the documents

In [3]:
for doc in dataset.docs_iter():
    print(doc) # namedtuple<doc_id, title, text, author, bib>
    break

CranfieldDoc(doc_id='1', title='experimental investigation of the aerodynamics of a\nwing in a slipstream .', text='experimental investigation of the aerodynamics of a\nwing in a slipstream .\n  an experimental study of a wing in a propeller slipstream was\nmade in order to determine the spanwise distribution of the lift\nincrease due to slipstream at different angles of attack of the wing\nand at different free stream to slipstream velocity ratios .  the\nresults were intended in part as an evaluation basis for different\ntheoretical treatments of this problem .\n  the comparative span loading curves, together with\nsupporting evidence, showed that a substantial part of the lift increment\nproduced by the slipstream was due to a /destalling/ or\nboundary-layer-control effect .  the integrated remaining lift\nincrement, after subtracting this destalling lift, was found to agree\nwell with a potential flow theory .\n  an empirical evaluation of the destalling effects was made for\nthe s

### Inspect the query-document relevance judgments

In [4]:
for qrel in dataset.qrels_iter():
    print(qrel) # namedtuple<query_id, doc_id, relevance, iteration>
    break

TrecQrel(query_id='1', doc_id='184', relevance=2, iteration='0')


## Setup

The cells below set up the imports and define the BM25 skeleton class. 
You do not need to modify anything here — simply run both cells before starting Task 1.

In [5]:
from __future__ import annotations

import math
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple, Set, Optional

import numpy as np

# NLTK for tokenization
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download("stopwords", quiet=True)

True

**Note:** The `BM25` class below contains all the methods you will implement throughout this assignment.
Each task will ask you to implement one or more of these methods and attach them to the class.

In [6]:
# BM25 Skeleton (run this first, do not edit)

class BM25:
    """
    BM25 class:
      - tokenizer: tokenize(text) -> List[str]
      - retriever: retrieve(query_text, k) -> ranked list
      - evaluator: evaluate(metric, k) and grid_search over k1, b

    Brute-force scoring over all documents (no inverted index).
    """
    def __init__(self, k1: float = 1.2, b: float = 0.75, remove_stopwords: bool = True):
        
        # --- BM25 Hyperparameters ---
        self.k1: float = k1                    # term frequency saturation
        self.b: float = b                      # document length normalization
        self.remove_stopwords: bool = remove_stopwords
        
        # Initializations for tokenization
        self._stemmer = PorterStemmer()      # work stemming
        self._stopwords = set(stopwords.words("english")) if self.remove_stopwords else set() # stopwords set
        
        # --- Corpus Statistics (populated by build()) ---
        self.N: int = 0                        # total number of documents
        self.avgdl: float = 0.0               # average document length
        self.df: Dict[str, int] = {}          # term -> number of docs containing term
        self.doc_len: Dict[str, int] = {}     # doc_id -> document length
        self.doc_tf: Dict[str, Counter] = {}  # doc_id -> term frequency Counter
        self.docs_store: Dict[str, Dict[str, str]] = {}  # doc_id -> {"title": ..., "text": ...}

    def tokenize(self, text: str) -> List[str]: raise NotImplementedError
    def build(self, docs: Iterable) -> None: raise NotImplementedError
    def idf(self, term: str) -> float: raise NotImplementedError
    def score(self, query_tokens: List[str], doc_id: str) -> float: raise NotImplementedError
    def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]: raise NotImplementedError
    def evaluate(self, queries_iter: Iterable, qrels_iter: Iterable, k: int = 10, metrics: Optional[List[str]] = None) -> Dict[str, float]: raise NotImplementedError
    def grid_search(self, queries_iter_factory, qrels_iter_factory, k: int = 10) -> List[Dict]: raise NotImplementedError



## Task 1: Tokenizer 

Implement a tokenizer that converts a text into a list of keywords: tokenize(text) -> List[str].

In [7]:
def tokenize(self, text: str) -> List[str]:
    """
    Convert raw text into a list of processed tokens.

    Requirements:
      1. Lowercase everything         e.g.  "UZH"                       -> ["uzh"]
      2. Extract alphanumeric words   e.g.  "hello, world 2026!"        -> ["hello", "world", "2026"]
      3. Remove stopwords             e.g.  "the document has a title"  -> ["document", "title"]
      4. Apply Porter stemming        e.g.  "working worked works"      -> ["work", "work", "work"]
      5. Drop empty tokens            e.g.  " retrieval "               -> ["retrieval"]

    Args:
        text: raw input string
    Returns:
        List of processed token strings

    Hints:
        - Perform text normalization using Python string methods.
        - Use the `re` module to extract alphanumeric tokens.
        - For stemming and stopword removal, use the initialized stemmer and stopword list from the BM25 class constructor.

    """

    text = text.lower()
    text = re.sub(r'[\W_]', ' ', text)
    tokenized_text: list[str] = [word for word in text.split() if word not in self._stopwords]
    return [self._stemmer.stem(word) for word in tokenized_text]
    
# attach to BM25 class
BM25.tokenize = tokenize

In [8]:
# ===============================================
# NOTE: Personal Testing Only — NOT Part of Assignment
# ===============================================
#
# This cell contains some basic example test cases for the `tokenize()` method.
# It is meant to help you check your own implementation.
# 
# It does NOT count toward your grade and is safe to run multiple times.
# You can modify it to test additional examples if you want.


bm25_test = BM25(remove_stopwords=True)

# Test 1: Lowercasing + stemming
assert bm25_test.tokenize("Working worked works") == ["work", "work", "work"]

# Test 2: Alphanumeric extraction + punctuation removal
assert bm25_test.tokenize("Hello, world 2026!") == ["hello", "world", "2026"]

# Test 3: Stopword removal + stemming
assert bm25_test.tokenize("The document has a title") == ["document", "titl"]

# Test 4: Mixed case + symbols + stemming
assert bm25_test.tokenize("Machine-Learning @ UZH :)") == ["machin", "learn", "uzh"]

print("All tokenizer tests passed")

All tokenizer tests passed


## Task 2: BM25 Retriever

Implement the document retriever using BM25 that finds the best top-k documents for a given query.

### Task 2a: Build Index

Implement `build()` to precompute the corpus statistics needed for BM25 scoring.

Once complete, the following attributes will be populated and used by `score()` and `retrieve()` in the next steps.

In [9]:
from collections import Counter, defaultdict


def build(self, docs_iter: Iterable) -> None:
    """
    Iterate over all documents and precompute corpus statistics needed for BM25.

    Args:
        docs_iter: iterable of Cranfield documents, each a namedtuple with fields:
                   (doc_id, title, text, author, bib)
    
    Requirements: 
        After calling build(), the following must be populated:
            self.N        : int                      — total number of documents
            self.avgdl    : float                    — average document length
            self.df       : Dict[str, int]           — term -> number of docs containing term
            self.doc_len  : Dict[str, int]           — doc_id -> document length
            self.doc_tf   : Dict[str, Counter]       — doc_id -> term frequencies per document
            self.docs_store: Dict[str, Dict[str, str]] — doc_id:str -> {"title": ..., "text": ...} (raw title & text per document)

    Note:
        Per-document steps (repeat for each document):
            1. Extract title and text safely
            2. Concatenate title and text, then tokenize using self.tokenize()
            3. Compute term frequencies  → store in self.doc_tf
            4. Compute document length → store in self.doc_len
            5. Store raw title and text → store in self.docs_store

        Corpus-level steps (after all documents):
            6. Count how many documents each term appears in → store in self.df
            (each term is counted once per document, not once per occurrence)
            7. Compute self.avgdl and self.N

    """
    df_counter = defaultdict(int)
    total_len = 0

    for doc in docs_iter:
        self.N += 1

        title = doc.title if doc.title is not None else ""
        text = doc.text if doc.text is not None else ""
        tokens = self.tokenize(" ".join([title, text]))

        tf = Counter(tokens)
        doc_id = (doc.doc_id)

        self.doc_tf[doc_id] = tf
        self.doc_len[doc_id] = len(tokens)
        self.docs_store[doc_id] = {"title": title, "text": text}

        total_len += len(tokens)

        for term in tf:
            df_counter[term] += 1

    self.df = dict(df_counter)
    self.avgdl = total_len / self.N if self.N > 0 else 0.0

# attach to BM25 class
BM25.build = build

### Task 2b: BM25 Scoring

Now that the corpus statistics can be precomputed, implement the two core scoring functions of BM25.

- `idf(term)` computes how rare a term is across the corpus
- `score(query_tokens, doc_id)` combines IDF with term frequency to compute a BM25 score for a query-document pair


In [10]:
def idf(self, term: str) -> float:
    """
    Compute Inverse Document Frequency with standard smoothing.

    Args:
        term: a single token str

    Returns:
        float — IDF score for the term

    Formula:
        log( (N - df + 0.5) / (df + 0.5) + 1 )

    """
    df = self.df.get(term, 0)
    return math.log((self.N-df+0.5)/(df+0.5)+1)

# attach to BM25 class
BM25.idf = idf


def score(self, query_tokens: List[str], doc_id: str) -> float:
    """
    Compute BM25 score for a query against a single document.

    Args:
        query_tokens: list of tokenized query terms
        doc_id: document to score against

    Returns:
        float — BM25 score for this query-document pair

    Formula (sum over query terms):
        score(q, d) = sum over t in q [ IDF(t) * (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl/avg_dl)) ]

    where:
        q    = query
        d    = document
        t    = a query term
        f    = term frequency of t in document
        dl   = length of the document
        k1   = self.k1
        b    = self.b

    Hints:
        - skip terms with frequency 0
        - return 0.0 for empty documents
        - sum over query terms (you can keep duplicates, but set() is common and stable for students)
    """

    dl = self.doc_len.get(doc_id, 0)
    if dl == 0: return 0

    total = 0
    for term in set(query_tokens):
        tf = self.doc_tf[doc_id].get(term, 0)
        if tf == 0: continue
        total += self.idf(term) * (tf*(self.k1+1)) / (tf+self.k1*(1-self.b+self.b*dl/self.avgdl))
    return total


# attach to BM25 class
BM25.score = score

### Task 2c: Retrieve

Implement `retrieve()` to return the top-k most relevant documents for a given query, ranked by BM25 score.

In [11]:
def retrieve(self, query_text: str, k: int = 10) -> List[Tuple[str, float]]:
    """
    Return the top-k documents for a given query, ranked by BM25 score.
    Brute-force over all documents.
    
    Args:
        query_text: raw query string
        k:          number of documents to return

    Returns:
        List of (doc_id, score) tuples sorted by score descending

    Hints:
        - tokenize the query first using self.tokenize()
        - iterate over all documents in self.doc_tf.keys()
        - use self.score() to compute the BM25 score for each document
        - only keep documents with a non-zero score
    """
    score_tuple_list = []

    query_tokenized = self.tokenize(query_text)
    for doc_id in self.doc_tf.keys(): 
        score = self.score(query_tokenized, doc_id)
        if score > 0:
            score_tuple_list.append((doc_id, score))
    score_tuple_list.sort(key=lambda x: x[1], reverse=True)
    return score_tuple_list[:k]

# attach to BM25 class
BM25.retrieve = retrieve

### Check Task 2: Implementing a BM25 retriever

Run the cell below to check your implementation. You should see:
- Total number of documents and average document length after building the index
- A ranked list of 10 documents for a sample query with their BM25 scores and titles


In [12]:
# Build index
bm25 = BM25()
print("Building index...")
bm25.build(dataset.docs_iter())
print(f"Done. N={bm25.N} docs, avgdl={bm25.avgdl:.2f}")

print('-'*120)

# Single query demo
q = next(dataset.queries_iter())
top10 = bm25.retrieve(q.text, k=10)
print("Query ID:", q.query_id, "Query Text:", q.text)
print('-'*50)
print("rank", "doc_id",  "score",  "title")
print('-'*50)
for rank, (doc_id, score) in enumerate(top10, 1):
    title = bm25.docs_store[doc_id]["title"]
    print(f"{rank:02d} {doc_id}  {score:.4f}  {title[:80]}")

Building index...
Done. N=1400 docs, avgdl=102.94
------------------------------------------------------------------------------------------------------------------------
Query ID: 1 Query Text: what similarity laws must be obeyed when constructing aeroelastic models
of heated high speed aircraft .
--------------------------------------------------
rank doc_id score title
--------------------------------------------------
01 51  21.9151  theory of aircraft structural models subjected to aerodynamic
heating and extern
02 486  21.2723  similarity laws for aerothermoelastic testing .
03 12  18.5659  some structural and aerelastic considerations of high
speed flight .
04 184  17.8901  scale models for thermo-aeroelastic research .
05 573  17.0003  viscous hypersonic similitude .
06 878  16.5020  experimental model techniques and equipment for flutter
investigations .
07 665  14.4273  on the theory of hypersonic gas flow with a power law
shock wave .
08 746  14.1964  aeroelastic problems in

## Task 3: BM25 Evaluator

### Task 3a: Implement evaluate function for "recall", "precision", "mrr", "ndcg"

In [13]:
def evaluate(
    self,
    queries_iter: Iterable,
    qrels_iter: Iterable,
    k: int = 10,
    metrics: Optional[List[str]] = None,
) -> Dict[str, float]:
    """
    Compute evaluation metrics averaged over all queries.

    Args:
        queries_iter : iterable of queries (each has .query_id and .text)
        qrels_iter   : iterable of relevance judgments (each has .query_id, .doc_id, .relevance)
        k            : number of top documents to evaluate
        metrics      : list of metrics to compute, e.g. ["recall", "precision", "mrr", "ndcg"]
                       if None, compute all four

    Returns:
        Dict mapping metric name -> average score across all queries

    Note:
        Relevance:
            - treat relevance <= 0 as non-relevant (0)
            - treat relevance  > 0 as relevant (1)

        Metrics:
            Recall@k    = (number of relevant docs in top-k) / (total relevant docs for query)

            Precision@k = (number of relevant docs in top-k) / k

            MRR@k       = 1 / (rank of first relevant doc in top-k)
                        0 if no relevant doc found

            nDCG@k      = DCG@k / IDCG@k
                        DCG@k  = sum over rank i: rel_i / log2(i + 2)    (i is 0-indexed)
                        IDCG@k = DCG of the ideal ranking (sort all relevant docs by relevance desc)

    Hints:
        - define two helper functions inside evaluate():
            * normalize_relevance(rel) to convert relevance scores to 0 or 1 (binary)
            * dcg(rels) to compute DCG for a ranked relevance list
        - build a qrels dict: qrels[query_id][doc_id] = normalized relevance
        - skip queries that have no relevance judgments
        - use self.retrieve(query_text, k) to get ranked (doc_id, score) pairs
        - for duplicate qrel entries keep the maximum relevance
    """

    if metrics is None:
        metrics = ["recall", "precision", "mrr", "ndcg"]

    def normalize_relevance(rel):
        return 1 if rel > 0 else 0

    def dcg(rels):
        return sum(rel / math.log2(i + 2) for i, rel in enumerate(rels))

    qrels = defaultdict(dict)
    for qrel in qrels_iter:
        query_id = qrel.query_id
        doc_id = qrel.doc_id
        rel = normalize_relevance(qrel.relevance)
        qrels[query_id][doc_id] = max(qrels[query_id].get(doc_id, 0), rel)

    totals = {metric: 0.0 for metric in metrics}
    evaluated_queries = 0

    for query in queries_iter:
        query_id = query.query_id
        if query_id not in qrels:
            continue

        query_qrels = qrels[query_id]
        ranked_docs = self.retrieve(query.text, k)
        ranked_rels = [query_qrels.get(doc_id, 0) for doc_id, _ in ranked_docs]

        relevant_in_top_k = sum(ranked_rels)
        total_relevant = sum(query_qrels.values())

        if "recall" in totals:
            totals["recall"] += relevant_in_top_k / total_relevant if total_relevant > 0 else 0.0

        if "precision" in totals:
            totals["precision"] += relevant_in_top_k / k if k > 0 else 0.0

        if "mrr" in totals:
            reciprocal_rank = 0.0
            for rank, rel in enumerate(ranked_rels, start=1):
                if rel > 0:
                    reciprocal_rank = 1.0 / rank
                    break
            totals["mrr"] += reciprocal_rank

        if "ndcg" in totals:
            ideal_rels = sorted(query_qrels.values(), reverse=True)[:k]
            ideal_dcg = dcg(ideal_rels)
            totals["ndcg"] += dcg(ranked_rels) / ideal_dcg if ideal_dcg > 0 else 0.0

        evaluated_queries += 1

    if evaluated_queries == 0:
        return {metric: 0.0 for metric in metrics}
    
    return {metric: totals[metric] / evaluated_queries for metric in metrics}

BM25.evaluate = evaluate

### Task 3b: Grid Search

Implement and perform grid search for k1 ∈ {0.60, 1.20, 1.80} and b ∈ {0.25, 0.50, 0.75}

In [14]:
def grid_search(
    self,
    queries_iter_factory,
    qrels_iter_factory,
    k: int = 10,
    k1_values: Tuple = (0.60, 1.20, 1.80),
    b_values: Tuple = (0.25, 0.50, 0.75),
) -> List[Dict]:
    """
    Evaluate all combinations of k1 and b, return results as a list of dicts.

    Args:
        queries_iter_factory : callable that returns a fresh queries iterator each time
        qrels_iter_factory   : callable that returns a fresh qrels iterator each time
        k                    : number of top documents to evaluate
        k1_values            : values of k1 to try
        b_values             : values of b to try

    Returns:
        List of dicts, one per (k1, b) combination:
        [
            {"k1": 0.60, "b": 0.25, "recall": ..., "precision": ..., "mrr": ..., "ndcg": ...},
            ...
        ]

    Note:
        - update self.k1 and self.b directly before each self.evaluate() call
        - pass factories/lambdas not iterators as ir_datasets iterators are single-pass,
          so call queries_iter_factory() and qrels_iter_factory() inside the loop to get a fresh iterator each time
    """

    grid_search_list = []

    for k1 in k1_values: 
        for b in b_values:
            self.k1 = k1
            self.b = b
            eval = self.evaluate(queries_iter_factory(), qrels_iter_factory(), k=k)
            grid_search_list.append({
                "k1": k1,
                "b": b,
                "recall": eval.get("recall"),
                "precision": eval.get("precision"),
                "mrr": eval.get("mrr"),
                "ndcg": eval.get("ndcg")
            })

    return grid_search_list
# attach to BM25 class
BM25.grid_search = grid_search

#### Run the cell below to perform the grid search

In [15]:
# Grid search
# Make sure you have run the Check Task 2 cell first to initialize bm25
print("\nRunning grid search...")
results = bm25.grid_search(
    queries_iter_factory=dataset.queries_iter,
    qrels_iter_factory=dataset.qrels_iter,
    k=10,
)

# Display results table
import pandas as pd

df = pd.DataFrame(results)
df = df.set_index(["k1", "b"])
df.columns = ["Recall@10", "Precision@10", "MRR@10", "nDCG@10"] # change if your metric order is different
print("\n", df.to_string(float_format="%.4f"))



Running grid search...

           Recall@10  Precision@10  MRR@10  nDCG@10
k1  b                                             
0.6 0.25     0.3745        0.2169  0.5173   0.3623
    0.50     0.3845        0.2231  0.5251   0.3734
    0.75     0.3896        0.2280  0.5229   0.3773
1.2 0.25     0.3922        0.2293  0.5307   0.3803
    0.50     0.4017        0.2382  0.5315   0.3899
    0.75     0.3992        0.2396  0.5353   0.3922
1.8 0.25     0.3976        0.2364  0.5360   0.3877
    0.50     0.4005        0.2409  0.5406   0.3930
    0.75     0.4086        0.2458  0.5417   0.3992


#### TODO: Report and justify your choice of best configuration



**Which do you consider your best configuration:** (k1: 1.8, b: 0.75)

**Why:** This configuration yields a globally optimal solution, i.e., each metric is best given these parameters eliminating the need for any potential trade-off considerations.

## Task 4: Error Analysis

### TODO:
### Pick two queries, one where BM25 retrieved great documents, one where BM25 failed the retrieval.


In [35]:
def rank_query_results():
    bm25.k1 = 1.8
    bm25.b = 0.75
    query_quality_metrics = []
    for query_obj in dataset.queries_iter():
        bm25_res = bm25.retrieve(query_obj.text, k=10)
        query_qrels = [qrel for qrel in dataset.qrels_iter() if qrel.query_id == query_obj.query_id]
        bm25_eval = bm25.evaluate([query_obj], query_qrels, k=10)
        bm25_eval["aggregate"] = (bm25_eval["recall"] + bm25_eval["precision"] + bm25_eval["mrr"] + bm25_eval["ndcg"])/4
        query_quality_metrics.append({
            "id": query_obj.query_id,
            "text": query_obj.text,
            "res": bm25_res,
            "eval": bm25_eval,
            "qrels": query_qrels
        })

    query_quality_metrics.sort(key=lambda x: x["eval"]["aggregate"], reverse=True)
    return query_quality_metrics

results = rank_query_results()


In [70]:
print("OVERVIEW")
print(20*"=", " BEST ", 20*"=")
top = results[0]
print("ID: ", top["id"])
print("Text: ", top["text"])
for i, elem in enumerate(top["res"]):
    relevance = next((x[2] for x in top["qrels"] if x[1] == elem[0]), None)
    print(i+1, "-- DOC ID:", elem[0], "-- BM25:", elem[1], " -- RELEVANCE:", relevance, " -- TITLE:", bm25.docs_store[elem[0]].get("title").replace("\n", " "))
    if i >= 4: break

print(20*"=", " WORST ", 20*"=")
worst = results[-1]
print("ID: ", worst["id"])
print("Text: ", worst["text"])
for i, elem in enumerate(worst["res"]):
    relevance = next((x[2] for x in worst["qrels"] if x[1] == elem[0]), None)
    print(i+1, "-- DOC ID:", elem[0], "-- BM25:", elem[1], " -- RELEVANCE:", relevance, " -- TITLE:", bm25.docs_store[elem[0]].get("title").replace("\n", " "))
    if i >= 4: break


OVERVIEW
====================  BEST  ====================
ID:  193
Text:  general methods of solving clamped plate problems .
1 -- DOC ID: 730 -- BM25: 20.383130103136494  -- RELEVANCE: 3  -- TITLE: on the bending of a clamped plate .
2 -- DOC ID: 733 -- BM25: 18.75988668425768  -- RELEVANCE: 4  -- TITLE: the bending of a sectorial plate .
3 -- DOC ID: 422 -- BM25: 17.728078335983906  -- RELEVANCE: 3  -- TITLE: bending of a square plate with two adjacent edges free and the others clamped or simply supported .
4 -- DOC ID: 734 -- BM25: 17.47493212388741  -- RELEVANCE: 4  -- TITLE: the bending of uniformly loaded clamped plate in the form of a circular sector .
5 -- DOC ID: 641 -- BM25: 16.570331655449177  -- RELEVANCE: -1  -- TITLE: reduction of the clamped plate to two membrane problems with an application to uniformly loaded sectors .
====================  WORST  ====================
ID:  219
Text:  what are the general effects on flow fields when the reynolds number is
small .
1 -- D

#### Query 1 — Successful Retrieval

I refer to the "BEST" section above for the Query ID, Query Text, Top 5 Retrieved Documents 

**Does the ranking make sense?** The ranking makes sense on a vocabulary level given both the relevance scores as well as the titles. While I am not an expert for clamped plate problems, the first four results are considered relevant by human annotators, while the 5th element is off topic, since it offers a specific solution to the problem rather than a general method as queried. Its evident that the first hit does not have the hightest relevance score. This is, however, rather negligible. The top ranks are dominated by high relevance scores (3 or 4) and these minor inversions and also in line with potential ambiguities in natural language.

**Proposed Improvement** There does exist one document, which is not considered relevant since it addresses the same problem but with a different angle. The alignment with BM25 is still significant, since on a vocabulary level the topics are highly related. In such cases, it might be necessary to consider the semantics on a document level rather than on a token level to separate different concerns.

#### Query 2 — Unsuccessful Retrieval

I refer to the "WORST" section above for the Query ID, Query Text, Top 5 Retrieved Documents 

**Does the ranking make sense?** As to be expected for the worst performing example, the ranking is not meaning full for IR. The query is composed of generic terms that carry little unique identifiable meaning. This is equivalent to a low IDF for the query, such that we cannot extract much meaningful information and we end up with numerous false positives. While we do have the distinct mention of "Reynolds number", it is tokenized separately, rather than as one standing term and might be Out-Of-Vocabulary. These factors lead to the eventual breakdown of BM25.


**Proposed Improvement** Again, using a meta level semantic approach, e.g., using neural embedding might remove some of the issues stemming from "brute force" tokenization. As a middle ground, between these two extremes, a more advanced tokenization approach might yield better results, e.g., using bi-grams - however, computationally very inefficient.